In [ ]:
```xml
<VSCode.Cell language="markdown">
# 💰 FinOps & Cost Optimization Guide

**Managing cloud costs, optimizing resource usage, and maximizing ROI through financial operations practices**

This guide covers:
- FinOps principles and practice
- Cost allocation and chargeback
- Reserved instances and savings plans
- Resource rightsizing
- Waste detection and elimination
- Budget forecasting
- Cloud cost optimization strategies
- Financial governance
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. FinOps Fundamentals

### Cost Tracking & Attribution
```python
from dataclasses import dataclass
from typing import Dict, List, Optional
from datetime import datetime, timedelta
from enum import Enum

class CostCategory(Enum):
    """Cloud cost categories"""
    COMPUTE = "compute"  # VMs, containers
    STORAGE = "storage"  # S3, databases
    NETWORK = "network"  # Data transfer
    DATABASE = "database"  # Managed DBs
    ANALYTICS = "analytics"  # Data warehouse, ML
    SECURITY = "security"  # SSL, monitoring
    SUPPORT = "support"  # Support plan

@dataclass
class CostRecord:
    """Individual cloud cost entry"""
    resource_id: str
    resource_type: str
    category: CostCategory
    cost_usd: float
    currency: str = "USD"
    timestamp: datetime = None
    tags: Dict = None
    region: str = None
    account_id: str = None
    usage_value: float = 0  # Quantity of resource used
    usage_unit: str = ""  # e.g., "hours", "GB"
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now()
        if self.tags is None:
            self.tags = {}

class CostAnalyzer:
    """Analyze cloud costs"""
    
    def __init__(self):
        self.cost_records: List[CostRecord] = []
        self.budgets: Dict[str, float] = {}  # category -> budget
    
    def record_cost(self, record: CostRecord):
        """Record cloud cost"""
        self.cost_records.append(record)
    
    def set_budget(self, category: str, budget_usd: float):
        """Set budget for category"""
        self.budgets[category] = budget_usd
    
    def get_total_cost(self, days: int = 30) -> float:
        """Calculate total cost for period"""
        cutoff = datetime.now() - timedelta(days=days)
        
        return sum(
            record.cost_usd for record in self.cost_records
            if record.timestamp >= cutoff
        )
    
    def get_cost_by_category(self, days: int = 30) -> Dict[str, float]:
        """Breakdown costs by category"""
        cutoff = datetime.now() - timedelta(days=days)
        
        costs_by_category = {}
        
        for record in self.cost_records:
            if record.timestamp >= cutoff:
                category = record.category.value
                
                if category not in costs_by_category:
                    costs_by_category[category] = 0
                
                costs_by_category[category] += record.cost_usd
        
        return costs_by_category
    
    def get_cost_by_tag(self, tag_key: str, days: int = 30) -> Dict[str, float]:
        """Breakdown costs by tag value (cost allocation)"""
        cutoff = datetime.now() - timedelta(days=days)
        
        costs_by_tag = {}
        
        for record in self.cost_records:
            if record.timestamp >= cutoff and record.tags:
                tag_value = record.tags.get(tag_key, "untagged")
                
                if tag_value not in costs_by_tag:
                    costs_by_tag[tag_value] = 0
                
                costs_by_tag[tag_value] += record.cost_usd
        
        return costs_by_tag
    
    def check_budget_status(self) -> Dict[str, Dict]:
        """Check budget compliance"""
        costs_by_category = self.get_cost_by_category(days=30)
        
        status = {}
        
        for category, budget in self.budgets.items():
            actual = costs_by_category.get(category, 0)
            percentage = (actual / budget * 100) if budget > 0 else 0
            
            status[category] = {
                'budget': budget,
                'actual': actual,
                'percentage': percentage,
                'status': 'under' if percentage < 80 else ('warning' if percentage < 100 else 'exceeded')
            }
        
        return status
    
    def forecast_costs(self, days_ahead: int = 30) -> Dict[str, float]:
        """Forecast future costs based on trends"""
        # Simple linear extrapolation
        recent_costs = self.get_cost_by_category(days=30)
        daily_rate = {k: v/30 for k, v in recent_costs.items()}
        
        forecast = {
            category: daily_rate.get(category, 0) * days_ahead
            for category in self.budgets.keys()
        }
        
        return forecast

# Usage
analyzer = CostAnalyzer()
analyzer.set_budget("compute", 5000)
analyzer.set_budget("storage", 1000)

# Record some costs
cost1 = CostRecord(
    resource_id="vm-prod-1",
    resource_type="t3.large",
    category=CostCategory.COMPUTE,
    cost_usd=150.50,
    tags={"environment": "production", "team": "backend"}
)

analyzer.record_cost(cost1)
print(analyzer.check_budget_status())
```

### Chargeback Models
```python
class ChargebackModel:
    """Model for cost allocation and chargeback"""
    
    def __init__(self, model_type: str):
        self.model_type = model_type  # "show-back", "chargeback", "fixed", "tiered"
        self.cost_pools: Dict[str, float] = {}
        self.allocations: Dict[str, Dict] = {}
    
    def add_cost_pool(self, pool_name: str, total_cost: float):
        """Add shared cost pool"""
        self.cost_pools[pool_name] = total_cost
    
    def allocate_showback(self, department: str, resource_usage: float, total_usage: float):
        """Show-back: show costs without charging"""
        if department not in self.allocations:
            self.allocations[department] = {}
        
        # Allocate proportionally
        shared_cost = sum(self.cost_pools.values())
        department_share = (resource_usage / total_usage) * shared_cost if total_usage > 0 else 0
        
        self.allocations[department]['showback'] = department_share
    
    def allocate_chargeback(self, department: str, resource_usage: float, total_usage: float):
        """Chargeback: charge departments for their usage"""
        if department not in self.allocations:
            self.allocations[department] = {}
        
        shared_cost = sum(self.cost_pools.values())
        department_charge = (resource_usage / total_usage) * shared_cost if total_usage > 0 else 0
        
        self.allocations[department]['chargeback'] = department_charge
    
    def get_department_allocation(self, department: str) -> Dict:
        """Get cost allocation for department"""
        return self.allocations.get(department, {})
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Resource Optimization

### Rightsizing & Waste Detection
```python
class ResourceProfile:
    """Profile of resource utilization"""
    
    def __init__(self, resource_id: str):
        self.resource_id = resource_id
        self.cpu_samples: List[float] = []  # Percentage
        self.memory_samples: List[float] = []
        self.network_samples: List[float] = []
        self.timestamp_samples: List[datetime] = []
    
    def add_sample(self, cpu: float, memory: float, network: float):
        """Add utilization sample"""
        self.cpu_samples.append(cpu)
        self.memory_samples.append(memory)
        self.network_samples.append(network)
        self.timestamp_samples.append(datetime.now())
    
    def get_average_cpu(self) -> float:
        """Get average CPU utilization"""
        return sum(self.cpu_samples) / len(self.cpu_samples) if self.cpu_samples else 0
    
    def get_average_memory(self) -> float:
        """Get average memory utilization"""
        return sum(self.memory_samples) / len(self.memory_samples) if self.memory_samples else 0
    
    def get_peak_cpu(self) -> float:
        """Get peak CPU utilization"""
        return max(self.cpu_samples) if self.cpu_samples else 0
    
    def is_underutilized(self, cpu_threshold: float = 0.1, memory_threshold: float = 0.1) -> bool:
        """Check if resource is underutilized"""
        return (self.get_average_cpu() < cpu_threshold and 
                self.get_average_memory() < memory_threshold)
    
    def suggest_rightsizing(self) -> Optional[str]:
        """Suggest smaller instance type"""
        avg_cpu = self.get_average_cpu()
        
        if avg_cpu < 0.05:
            return f"Consider stopping resource (avg CPU: {avg_cpu:.1%})"
        elif avg_cpu < 0.1:
            return f"Could downsize 1-2 sizes (avg CPU: {avg_cpu:.1%})"
        elif avg_cpu < 0.3:
            return f"Could downsize 1 size (avg CPU: {avg_cpu:.1%})"
        
        return None

class WasteDetector:
    """Detect cloud waste"""
    
    def __init__(self):
        self.profiles: Dict[str, ResourceProfile] = {}
        self.unattached_volumes: List[Dict] = []
        self.unused_ips: List[str] = []
    
    def analyze_resource(self, resource_id: str, profile: ResourceProfile) -> Dict:
        """Analyze resource for waste"""
        self.profiles[resource_id] = profile
        
        issues = []
        
        if profile.is_underutilized():
            issues.append(f"Underutilized (CPU avg: {profile.get_average_cpu():.1%})")
        
        suggestion = profile.suggest_rightsizing()
        if suggestion:
            issues.append(suggestion)
        
        return {
            'resource_id': resource_id,
            'avg_cpu': profile.get_average_cpu(),
            'peak_cpu': profile.get_peak_cpu(),
            'avg_memory': profile.get_average_memory(),
            'waste_issues': issues,
            'recommended_action': 'resize' if issues else 'keep'
        }
    
    def report_orphaned_resources(self) -> Dict:
        """Report unattached/unused resources"""
        return {
            'unattached_volumes': len(self.unattached_volumes),
            'unused_ip_addresses': len(self.unused_ips),
            'estimated_waste_usd_monthly': (
                len(self.unattached_volumes) * 10 +  # Assume $10/volume/month
                len(self.unused_ips) * 3  # Assume $3/IP/month
            )
        }
    
    def get_optimization_opportunities(self) -> List[Dict]:
        """Get list of optimization opportunities"""
        opportunities = []
        
        for resource_id, profile in self.profiles.items():
            if profile.is_underutilized():
                avg_cpu = profile.get_average_cpu()
                potential_savings = 0.6  # Assume 60% savings from rightsizing
                
                opportunities.append({
                    'resource': resource_id,
                    'type': 'rightsize',
                    'current_cost_estimate': 100,  # Placeholder
                    'potential_savings_usd': 100 * potential_savings,
                    'effort': 'low'
                })
        
        return opportunities
```

### Reserved Instances & Savings Plans
```python
class SavingsPlanning:
    """Plan for reserved instances and savings plans"""
    
    def __init__(self):
        self.on_demand_costs: Dict[str, float] = {}
        self.reserved_instances: Dict[str, Dict] = {}
    
    def register_on_demand_usage(self, resource_type: str, monthly_cost: float):
        """Register on-demand usage"""
        self.on_demand_costs[resource_type] = monthly_cost
    
    def calculate_ri_payback(self, resource_type: str, 
                            upfront_cost: float, 
                            discounted_rate: float = 0.37) -> Dict:
        """Calculate reserved instance payback"""
        on_demand_monthly = self.on_demand_costs.get(resource_type, 0)
        
        ri_monthly_cost = on_demand_monthly * (1 - discounted_rate)
        monthly_savings = on_demand_monthly - ri_monthly_cost
        payback_months = upfront_cost / monthly_savings if monthly_savings > 0 else float('inf')
        
        return {
            'resource_type': resource_type,
            'upfront_cost': upfront_cost,
            'on_demand_monthly': on_demand_monthly,
            'ri_monthly_cost': ri_monthly_cost,
            'monthly_savings': monthly_savings,
            'payback_months': payback_months,
            'first_year_savings': (monthly_savings * 12) - upfront_cost
        }
    
    def recommend_ri_purchases(self) -> List[Dict]:
        """Recommend RI purchases based on usage patterns"""
        recommendations = []
        
        for resource_type, on_demand_cost in self.on_demand_costs.items():
            # Estimate RI cost (typically 37% discount for 1-year)
            ri_upfront = on_demand_cost * 4  # Rough estimate
            
            payback = self.calculate_ri_payback(resource_type, ri_upfront)
            
            if payback['payback_months'] < 12 and payback['first_year_savings'] > 0:
                recommendations.append(payback)
        
        return sorted(recommendations, 
                     key=lambda x: x['first_year_savings'], 
                     reverse=True)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Financial Governance

### Cost Controls & Enforcement
```python
class CostPolicy:
    """Define cost control policies"""
    
    def __init__(self, name: str):
        self.name = name
        self.rules: List[Dict] = []
    
    def add_rule(self, rule_type: str, condition: str, action: str):
        """Add governance rule"""
        self.rules.append({
            'type': rule_type,
            'condition': condition,
            'action': action,
            'enabled': True
        })
    
    def enforce_instance_type_limits(self, allowed_types: List[str]):
        """Allow only specific instance types"""
        self.add_rule(
            'instance_restriction',
            f'instance_type NOT IN {allowed_types}',
            'block_launch'
        )
    
    def enforce_spot_instances(self, percentage: float = 0.8):
        """Require using spot instances for non-critical workloads"""
        self.add_rule(
            'spot_requirement',
            f'non_critical AND instance_type NOT spot',
            'block_on_demand_launch'
        )

class BudgetAlert:
    """Budget alerting system"""
    
    def __init__(self, threshold_percentage: float = 0.8):
        self.threshold = threshold_percentage
        self.alerts: List[Dict] = []
    
    def check_budget(self, category: str, spent: float, budget: float) -> bool:
        """Check if budget exceeded"""
        percentage = spent / budget if budget > 0 else 0
        
        if percentage >= self.threshold:
            severity = 'critical' if percentage >= 1.0 else 'warning'
            
            self.alerts.append({
                'category': category,
                'severity': severity,
                'percentage': percentage,
                'spent': spent,
                'budget': budget,
                'timestamp': datetime.now()
            })
            
            return True
        
        return False
    
    def get_recent_alerts(self, hours: int = 24) -> List[Dict]:
        """Get recent alerts"""
        cutoff = datetime.now() - timedelta(hours=hours)
        
        return [a for a in self.alerts if a['timestamp'] >= cutoff]
```

### Cost Reporting & Optimization
```python
class CostOptimizationReport:
    """Generate cost optimization report"""
    
    def __init__(self, analyzer: CostAnalyzer, detector: WasteDetector):
        self.analyzer = analyzer
        self.detector = detector
    
    def generate_executive_summary(self) -> Dict:
        """Generate high-level cost summary"""
        current_costs = self.analyzer.get_total_cost(days=30)
        forecast = self.analyzer.forecast_costs(days_ahead=30)
        total_forecast = sum(forecast.values())
        
        opportunities = self.detector.get_optimization_opportunities()
        potential_savings = sum(o['potential_savings_usd'] for o in opportunities)
        
        return {
            'current_monthly_cost': current_costs,
            'forecasted_next_month': total_forecast,
            'month_over_month_change': ((total_forecast - current_costs) / current_costs * 100) if current_costs > 0 else 0,
            'optimization_opportunities_count': len(opportunities),
            'estimated_monthly_savings': potential_savings,
            'estimated_annual_savings': potential_savings * 12
        }
    
    def generate_detailed_report(self) -> Dict:
        """Generate detailed optimization report"""
        summary = self.generate_executive_summary()
        costs_by_category = self.analyzer.get_cost_by_category()
        opportunities = self.detector.get_optimization_opportunities()
        orphaned = self.detector.report_orphaned_resources()
        
        return {
            'summary': summary,
            'costs_by_category': costs_by_category,
            'optimization_opportunities': opportunities,
            'orphaned_resources': orphaned,
            'recommendations': [
                {
                    'priority': 'high',
                    'action': 'Rightsize underutilized instances',
                    'estimated_savings': sum(o['potential_savings_usd'] for o in opportunities)
                },
                {
                    'priority': 'high',
                    'action': 'Purchase reserved instances',
                    'estimated_savings': 2000
                },
                {
                    'priority': 'medium',
                    'action': 'Delete orphaned resources',
                    'estimated_savings': orphaned['estimated_waste_usd_monthly']
                }
            ]
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. FinOps Checklist

✅ **Cost Visibility**
- [ ] All costs tracked
- [ ] Cost tagging implemented
- [ ] Attribution models defined
- [ ] Chargeback calculated
- [ ] Trends identified

✅ **Optimization**
- [ ] Underutilized resources identified
- [ ] Rightsizing opportunities assessed
- [ ] Reserved instances evaluated
- [ ] Spot instances utilized
- [ ] Orphaned resources cleaned up

✅ **Governance**
- [ ] Budgets established
- [ ] Policies enforced
- [ ] Alerts configured
- [ ] Approval workflows active
- [ ] Reviews scheduled

✅ **Culture**
- [ ] Team trained
- [ ] Cost awareness built
- [ ] Best practices shared
- [ ] Incentives aligned
- [ ] Continuous improvement

✅ **Technology**
- [ ] Tools integrated
- [ ] APIs connected
- [ ] Automation implemented
- [ ] Dashboards built
- [ ] Alerts working
</VSCode.Cell>
```